[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch03/NN_DL_ch03_RegularizationViz/NN_DL_ch03_RegularizationViz.ipynb)

# NN_DL_ch03_RegularizationViz

**Regularization Techniques Visualized**

Visualize L1/L2 penalty contours, dropout, bias-variance tradeoff, and early stopping.

In [ ]:
!pip install -q torch matplotlib scikit-learn

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Bias-Variance Tradeoff
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

np.random.seed(42)
x_true = np.linspace(0, 1, 200)
y_true = np.sin(2 * np.pi * x_true)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ['Underfitting (degree=1)', 'Good Fit (degree=4)', 'Overfitting (degree=15)']
degrees = [1, 4, 15]

for ax, deg, title in zip(axes, degrees, titles):
    ax.plot(x_true, y_true, 'k-', lw=2, label='True function', alpha=0.5)
    for trial in range(5):
        x_s = np.sort(np.random.uniform(0, 1, 15))
        y_s = np.sin(2 * np.pi * x_s) + np.random.randn(15) * 0.3
        poly = PolynomialFeatures(deg)
        X_poly = poly.fit_transform(x_s.reshape(-1, 1))
        model_lr = LinearRegression().fit(X_poly, y_s)
        y_pred = model_lr.predict(poly.transform(x_true.reshape(-1, 1)))
        ax.plot(x_true, y_pred, color=MAIN_BLUE, alpha=0.4, lw=1)
        if trial == 0:
            ax.scatter(x_s, y_s, color=IDA_RED, s=30, zorder=5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(-2, 2)

plt.suptitle('Bias-Variance Tradeoff', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch03_bias_variance')

In [ ]:
# L2/L1 Regularization Contours
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, title, constraint_type in zip(axes,
    ['L2 (Ridge) Regularization', 'L1 (Lasso) Regularization'],
    ['circle', 'diamond']):

    w1, w2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
    loss = (w1 - 2)**2 + 4 * (w2 - 1.5)**2
    ax.contour(w1, w2, loss, levels=15, colors=MAIN_BLUE, alpha=0.5, linewidths=1)

    if constraint_type == 'circle':
        theta = np.linspace(0, 2*np.pi, 100)
        r = 1.2
        ax.fill(r * np.cos(theta), r * np.sin(theta), color=FOREST, alpha=0.2)
        ax.plot(r * np.cos(theta), r * np.sin(theta), color=FOREST, lw=2)
        ax.scatter([0.75], [0.95], color=IDA_RED, s=100, zorder=5, marker='*')
    else:
        r = 1.5
        diamond_x = [r, 0, -r, 0, r]
        diamond_y = [0, r, 0, -r, 0]
        ax.fill(diamond_x, diamond_y, color=FOREST, alpha=0.2)
        ax.plot(diamond_x, diamond_y, color=FOREST, lw=2)
        ax.scatter([0], [1.5], color=IDA_RED, s=100, zorder=5, marker='*')

    ax.scatter([2], [1.5], color=AMBER, s=100, zorder=5, marker='o', label='Unconstrained min')
    ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
    ax.set_xlabel('$w_1$'); ax.set_ylabel('$w_2$')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.legend()

plt.tight_layout()
save_fig('nn_ch03_l2_contour')

In [ ]:
# Dropout Diagram
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def draw_network(ax, title, drop_mask=None):
    layers = [4, 6, 6, 3]
    x_pos = np.linspace(0.1, 0.9, len(layers))
    positions = []
    for i, (n, x) in enumerate(zip(layers, x_pos)):
        y_pos = np.linspace(0.15, 0.85, n)
        pos = [(x, y) for y in y_pos]
        positions.append(pos)

    for i in range(len(layers) - 1):
        for j, (x1, y1) in enumerate(positions[i]):
            if drop_mask and drop_mask[i][j]:
                continue
            for k, (x2, y2) in enumerate(positions[i+1]):
                if drop_mask and drop_mask[i+1][k]:
                    continue
                alpha_v = 0.15 if drop_mask else 0.3
                ax.plot([x1, x2], [y1, y2], color=MAIN_BLUE, alpha=alpha_v, lw=0.5)

    for i, layer_pos in enumerate(positions):
        for j, (x, y) in enumerate(layer_pos):
            if drop_mask and drop_mask[i][j]:
                ax.plot(x, y, 'o', markersize=18, color='lightgrey',
                       markeredgecolor='grey', markeredgewidth=1)
                ax.plot([x-0.015, x+0.015], [y-0.015, y+0.015], color='grey', lw=2)
                ax.plot([x-0.015, x+0.015], [y+0.015, y-0.015], color='grey', lw=2)
            else:
                color = MAIN_BLUE if i == 0 else (FOREST if i == len(layers)-1 else IDA_RED)
                ax.plot(x, y, 'o', markersize=18, color=color, markeredgecolor='black', markeredgewidth=1)

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.axis('off')

draw_network(axes[0], 'Standard Network (No Dropout)')

np.random.seed(42)
drop_mask = []
for i, n in enumerate([4, 6, 6, 3]):
    if i == 0 or i == 3:
        drop_mask.append([False] * n)
    else:
        drop_mask.append([np.random.random() < 0.3 for _ in range(n)])
draw_network(axes[1], 'With Dropout (p=0.3)', drop_mask)

plt.suptitle('Dropout Regularization', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch03_dropout_diagram')

In [ ]:
# Early Stopping Visualization
import torch.nn as nn

np.random.seed(42)
x_data = np.sort(np.random.uniform(-3, 3, 30))
y_data = np.sin(x_data) + np.random.randn(30) * 0.3

x_t = torch.FloatTensor(x_data).unsqueeze(1)
y_t = torch.FloatTensor(y_data).unsqueeze(1)

x_val = np.sort(np.random.uniform(-3, 3, 100))
y_val = np.sin(x_val) + np.random.randn(100) * 0.3
x_vt = torch.FloatTensor(x_val).unsqueeze(1)
y_vt = torch.FloatTensor(y_val).unsqueeze(1)

class OverfitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
net = OverfitNet()
opt = torch.optim.Adam(net.parameters(), lr=0.01)
train_losses, val_losses = [], []

for epoch in range(500):
    net.train()
    pred = net(x_t)
    loss = nn.MSELoss()(pred, y_t)
    opt.zero_grad(); loss.backward(); opt.step()
    train_losses.append(loss.item())

    net.eval()
    with torch.no_grad():
        val_losses.append(nn.MSELoss()(net(x_vt), y_vt).item())

best_epoch = np.argmin(val_losses)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_losses, color=MAIN_BLUE, lw=2, label='Train Loss')
ax.plot(val_losses, color=IDA_RED, lw=2, label='Validation Loss')
ax.axvline(best_epoch, color=FOREST, lw=2, ls='--', label=f'Best epoch = {best_epoch}')
ax.fill_betweenx([0, max(val_losses)], best_epoch, 500, alpha=0.1, color=CRIMSON, label='Overfitting zone')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Early Stopping: Train vs Validation Loss', fontweight='bold')
ax.legend(fontsize=12)
save_fig('nn_ch03_early_stopping')

## Summary

Visualized key regularization concepts: **bias-variance tradeoff**, **L1/L2 penalty contours** (L1 promotes sparsity), **dropout** (randomly zeroing neurons), and **early stopping** (halting before overfitting).